In [57]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, asc, count, sum, avg, round, when
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

spark = SparkSession.builder \
    .appName("PagilaTasks") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

db_url = "jdbc:postgresql://localhost:5432/pagila"
db_props = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

film = spark.read.jdbc(db_url, "film", properties=db_props)
actor = spark.read.jdbc(db_url, "actor", properties=db_props)
film_actor = spark.read.jdbc(db_url, "film_actor", properties=db_props)
category = spark.read.jdbc(db_url, "category", properties=db_props)
film_category = spark.read.jdbc(db_url, "film_category", properties=db_props)
inventory = spark.read.jdbc(db_url, "inventory", properties=db_props)
rental = spark.read.jdbc(db_url, "rental", properties=db_props)
payment = spark.read.jdbc(db_url, "payment", properties=db_props)
customer = spark.read.jdbc(db_url, "customer", properties=db_props)
address = spark.read.jdbc(db_url, "address", properties=db_props)
city = spark.read.jdbc(db_url, "city", properties=db_props)

In [5]:
film.printSchema()
film_category.printSchema()
category.printSchema()

root
 |-- film_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- language_id: integer (nullable = true)
 |-- original_language_id: integer (nullable = true)
 |-- rental_duration: short (nullable = true)
 |-- rental_rate: decimal(4,2) (nullable = true)
 |-- length: short (nullable = true)
 |-- replacement_cost: decimal(5,2) (nullable = true)
 |-- rating: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- special_features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- fulltext: string (nullable = true)

root
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [8]:
df1 = film.join(film_category, on="film_id", how="inner").join(category, on="category_id", how="inner")
df1.printSchema()

root
 |-- category_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- language_id: integer (nullable = true)
 |-- original_language_id: integer (nullable = true)
 |-- rental_duration: short (nullable = true)
 |-- rental_rate: decimal(4,2) (nullable = true)
 |-- length: short (nullable = true)
 |-- replacement_cost: decimal(5,2) (nullable = true)
 |-- rating: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- special_features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- fulltext: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [11]:
cat_count = df1.groupBy("name").agg(count("*").alias("count")).orderBy(col("count").desc())
cat_count.show()

+-----------+-----+
|       name|count|
+-----------+-----+
|     Sports|   74|
|    Foreign|   73|
|     Family|   69|
|Documentary|   68|
|  Animation|   66|
|     Action|   64|
|        New|   63|
|      Drama|   62|
|      Games|   61|
|     Sci-Fi|   61|
|   Children|   60|
|     Comedy|   58|
|     Travel|   57|
|   Classics|   57|
|     Horror|   56|
|      Music|   51|
+-----------+-----+



In [5]:

actor.printSchema()
film_actor.printSchema()
rental.printSchema()
inventory.printSchema()

root
 |-- actor_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- actor_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- inventory_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [8]:
df2 = actor.join(film_actor, on="actor_id", how="inner").join(inventory, on="film_id", how="inner").join(rental, on="inventory_id", how="inner")
df2.printSchema()

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- actor_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [14]:
act_rent_count = df2.groupBy("actor_id", "first_name", "last_name").agg(count("*").alias("count")).orderBy(col("count").desc()).limit(10)
act_rent_count.show()

+--------+----------+-----------+-----+
|actor_id|first_name|  last_name|count|
+--------+----------+-----------+-----+
|     107|      GINA|  DEGENERES|  753|
|     181|   MATTHEW|     CARREY|  678|
|     198|      MARY|     KEITEL|  674|
|     144|    ANGELA|WITHERSPOON|  654|
|     102|    WALTER|       TORN|  640|
|      60|     HENRY|      BERRY|  612|
|     150|     JAYNE|      NOLTE|  611|
|      37|       VAL|     BOLGER|  605|
|      23|    SANDRA|     KILMER|  604|
|      90|      SEAN|    GUINESS|  599|
+--------+----------+-----------+-----+



category.printSchema()
film_category.printSchema()
rental.printSchema()
inventory.printSchema()
payment.printSchema()

In [19]:
df3 = category.join(film_category, on="category_id", how="inner").join(inventory, on="film_id", how="inner").join(rental, on="inventory_id", how="inner").join(payment, on="rental_id", how="inner")
df3.printSchema()

root
 |-- rental_id: integer (nullable = true)
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- amount: decimal(5,2) (nullable = true)
 |-- payment_date: timestamp (nullable = true)



In [25]:
cat_sum = df3.groupBy("name").agg(sum("amount").alias("spent")).orderBy(col("spent").desc()).limit(1)
cat_sum.show()

+------+-------+
|  name|  spent|
+------+-------+
|Sports|5314.21|
+------+-------+



In [26]:
film.printSchema()
inventory.printSchema()

root
 |-- film_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- language_id: integer (nullable = true)
 |-- original_language_id: integer (nullable = true)
 |-- rental_duration: short (nullable = true)
 |-- rental_rate: decimal(4,2) (nullable = true)
 |-- length: short (nullable = true)
 |-- replacement_cost: decimal(5,2) (nullable = true)
 |-- rating: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- special_features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- fulltext: string (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [30]:
df4 = film.join(inventory, on="film_id", how="left_anti").select("title")
df4.printSchema()

root
 |-- title: string (nullable = true)



In [31]:
df4.show()

+--------------------+
|               title|
+--------------------+
|      CHOCOLATE DUCK|
|       BUTCH PANTHER|
|        VOLUME HOUSE|
|      ORDER BETRAYED|
|        TADPOLE PARK|
|    KILL BROTHERHOOD|
|FRANKENSTEIN STRA...|
|    CROSSING DIVORCE|
|    SUICIDES SILENCE|
|       CATCH AMISTAD|
|     PERDITION FARGO|
|       FLOATS GARDEN|
|           GUMP DATE|
|        WALLS ARTIST|
|  GLADIATOR WESTWARD|
|         HOCUS FRIDA|
|ARSENIC INDEPENDENCE|
|         MUPPET MILE|
|   FIREHOUSE VIETNAM|
|       ROOF CHAMPION|
+--------------------+
only showing top 20 rows


In [34]:
film.printSchema()
film_category.printSchema()
category.printSchema()
actor.printSchema()
film_actor.printSchema()

root
 |-- film_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- language_id: integer (nullable = true)
 |-- original_language_id: integer (nullable = true)
 |-- rental_duration: short (nullable = true)
 |-- rental_rate: decimal(4,2) (nullable = true)
 |-- length: short (nullable = true)
 |-- replacement_cost: decimal(5,2) (nullable = true)
 |-- rating: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- special_features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- fulltext: string (nullable = true)

root
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- actor_id: integer (nullable = true)
 |-- first_na

In [37]:
df5 = category.join(film_category, on="category_id", how="inner").join(film_actor, on="film_id", how="inner").join(actor, on="actor_id", how="inner")
df5.printSchema()

root
 |-- actor_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [50]:
actor_child_cat = df5.filter(col("name")=="Children").groupBy("actor_id", "first_name",  "last_name").agg(count("*").alias("count"))
window = Window.orderBy(col("count").desc())
ranked_act = actor_child_cat.withColumn("rank", dense_rank().over(window))
res5 = ranked_act.filter(col("rank") <=3)
res5.show()

+--------+----------+---------+-----+----+
|actor_id|first_name|last_name|count|rank|
+--------+----------+---------+-----+----+
|      17|     HELEN|   VOIGHT|    7|   1|
|     127|     KEVIN|  GARLAND|    5|   2|
|      80|     RALPH|     CRUZ|    5|   2|
|      66|      MARY|    TANDY|    5|   2|
|     140|    WHOOPI|     HURT|    5|   2|
|      81|  SCARLETT|    DAMON|    4|   3|
|     109| SYLVESTER|     DERN|    4|   3|
|      23|    SANDRA|   KILMER|    4|   3|
|     187|     RENEE|     BALL|    4|   3|
|      92|   KIRSTEN|   AKROYD|    4|   3|
|     173|      ALAN| DREYFUSS|    4|   3|
|     101|     SUSAN|    DAVIS|    4|   3|
|     150|     JAYNE|    NOLTE|    4|   3|
|      13|       UMA|     WOOD|    4|   3|
|     131|      JANE|  JACKMAN|    4|   3|
|      58| CHRISTIAN|   AKROYD|    4|   3|
|     142|      JADA|    RYDER|    4|   3|
|      93|     ELLEN|  PRESLEY|    4|   3|
|      37|       VAL|   BOLGER|    4|   3|
+--------+----------+---------+-----+----+



In [53]:
customer.printSchema()
city.printSchema()
address.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address_id: integer (nullable = true)
 |-- activebool: boolean (nullable = true)
 |-- create_date: date (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- active: integer (nullable = true)

root
 |-- city_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- country_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- address_id: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- address2: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [54]:
df6 = customer.join(address, on="address_id", how="inner").join(city, on="city_id", how="inner")
df6.printSchema()

root
 |-- city_id: integer (nullable = true)
 |-- address_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- activebool: boolean (nullable = true)
 |-- create_date: date (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- active: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- address2: string (nullable = true)
 |-- district: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [58]:
cust_count = df6.groupBy("city").agg(count(when(col("active")==1, True)).alias("active"), count(when(col("active")==0, True)).alias("inactive")).orderBy(col("inactive").desc())
cust_count.show()

+------------------+------+--------+
|              city|active|inactive|
+------------------+------+--------+
|          Uluberia|     0|       1|
|         Najafabad|     0|       1|
|         Pingxiang|     0|       1|
|          Xiangfan|     0|       1|
|        Kumbakonam|     0|       1|
|       Szkesfehrvr|     0|       1|
|  Charlotte Amalie|     0|       1|
|            Kamyin|     0|       1|
|            Daxian|     0|       1|
|     Coatzacoalcos|     0|       1|
|           Wroclaw|     0|       1|
|            Ktahya|     0|       1|
|            Amroha|     0|       1|
|   Southend-on-Sea|     0|       1|
|           Bat Yam|     0|       1|
|          Fengshan|     1|       0|
|A Corua (La Corua)|     1|       0|
|           El Alto|     1|       0|
|              Linz|     1|       0|
|          Myingyan|     1|       0|
+------------------+------+--------+
only showing top 20 rows


In [63]:
rental.printSchema()
inventory.printSchema()
film_category.printSchema()
category.printSchema()

root
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- inventory_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [64]:
df7 = category.join(film_category, on="category_id", how="inner").join(inventory, on="film_id", how="inner").join(rental, on="inventory_id", how="inner")
df7.printSchema()

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)



In [65]:
hours_calc = (col("return_date").cast("long") - col("rental_date").cast("long")) / 3600
res7 = df7.filter((col("return_date").isNotNull()) & (col("name").startswith("A"))).groupBy("name").agg(sum(hours_calc).alias("total_rental_hours")).orderBy(col("total_rental_hours").desc())
res7.show()

+---------+------------------+
|     name|total_rental_hours|
+---------+------------------+
|Animation| 133885.9666666669|
|   Action|132104.69999999984|
+---------+------------------+



In [66]:
df8 = category.join(film_category, on="category_id", how="inner").join(inventory, on="film_id", how="inner").join(rental, on="inventory_id", how="inner").join(customer, on="customer_id", how="inner").join(address, on="address_id", how="inner").join(city, on="city_id", how="inner")

In [68]:
res8 = df8.filter((col("return_date").isNotNull()) & (col("city").contains("-"))).groupBy("name").agg(sum(hours_calc).alias("total_rental_hours")).orderBy(col("total_rental_hours").desc())
res8.show()

+-----------+------------------+
|       name|total_rental_hours|
+-----------+------------------+
|    Foreign| 6472.150000000001|
|     Action|6212.4333333333325|
|     Sports| 6126.916666666665|
|     Sci-Fi| 6033.183333333333|
|      Music| 5763.966666666665|
|      Drama| 5261.633333333336|
|     Comedy| 5212.416666666667|
|  Animation| 4974.933333333332|
|        New| 4866.066666666665|
|     Family| 4781.416666666666|
|   Classics| 4735.800000000002|
|Documentary| 4694.633333333332|
|   Children|4598.2333333333345|
|      Games|            4195.0|
|     Horror|3788.4500000000003|
|     Travel|3313.2833333333333|
+-----------+------------------+

